<a href="https://www.kaggle.com/code/alexvmt/terainet-inference-example?scriptVersionId=335791988" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/alexvmt/terainet-inference-example?scriptVersionId=247707904" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# TeraiNet inference example

Follow [mewc-predict](https://github.com/zaandahl/mewc-predict/blob/main/src/mewc_predict.py) for the general procedure

## Setup

### Imports

Follow [mewc-flow](https://github.com/zaandahl/mewc-flow/blob/main/requirements.txt) for the key package versions

In [1]:
!pip install keras==3.3.3 kimm==0.2.5 tensorflow==2.16.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.4/123.4 kB 6.0 MB/s eta 0:00:00


In [2]:
import os
import random
import shutil
from pathlib import Path

import pandas as pd
import tensorflow as tf
import yaml
from keras import saving

### Utilities

In [3]:
def copy_random_images(src_dir, dst_dir, n, seed=42):
    """
    Copies n random images from src_dir to dst_dir.

    Args:
        src_dir (str): Source directory containing images.
        dst_dir (str): Destination directory to copy images into.
        n (int): Number of images to copy.
        seed (int): Random seed for reproducibility.
    """
    # Ensure target directory exists
    os.makedirs(dst_dir, exist_ok=True)

    # List all files in the source directory
    all_files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]

    # Check if n is greater than available files
    if n > len(all_files):
        raise ValueError(
            f"Requested {n} images, but only {len(all_files)} available in source directory."
        )

    # Randomly sample n files
    random.seed(seed)
    selected_files = random.sample(all_files, n)

    # Copy each file to the target directory
    for filename in selected_files:
        src_path = os.path.join(src_dir, filename)
        dst_path = os.path.join(dst_dir, filename)
        shutil.copy2(src_path, dst_path)

    print(f"Copied {n} random images from '{src_dir}' to '{dst_dir}'.")

## Prepare images

In [4]:
copy_random_images("../input/preprocess-images/terainet_images/test2/class_1", "../images", 10)

Copied 10 random images from '../input/preprocess-images/terainet_images/test2/class_1' to '../images'.


In [5]:
img_generator = tf.keras.preprocessing.image_dataset_from_directory(
    "../images", labels=None, label_mode=None, batch_size=8, image_size=(224, 224), shuffle=False
)

Found 10 files.


## Predict

In [6]:
model = saving.load_model("../input/train-and-evaluate-terainet/model.keras", compile=False)

In [7]:
preds = model.predict(img_generator)

I0000 00:00:1784231672.364827      77 service.cc:145] XLA service 0x7bc8280032b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784231672.364898      77 service.cc:153]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0


1/2 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step

I0000 00:00:1784231679.708629      77 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2/2 ━━━━━━━━━━━━━━━━━━━━ 23s 11s/step


In [8]:
preds

array([[0.84674287, 0.02370572, 0.01107171, 0.01372842, 0.01831189,
        0.01507596, 0.02201088, 0.02069564, 0.01198735, 0.01666968],
       [0.00792261, 0.03121234, 0.7617105 , 0.02386605, 0.02557292,
        0.03509789, 0.02906937, 0.01509295, 0.06149056, 0.00896472],
       [0.740985  , 0.06773984, 0.02880301, 0.02516732, 0.03688958,
        0.01440554, 0.02417418, 0.01989504, 0.01772652, 0.024214  ],
       [0.8303648 , 0.02398647, 0.01483363, 0.02067083, 0.02197562,
        0.02233409, 0.01618276, 0.01863491, 0.01590108, 0.01511574],
       [0.72691184, 0.08180103, 0.02423305, 0.03061306, 0.02657528,
        0.03416738, 0.02031484, 0.01013856, 0.02785124, 0.01739373],
       [0.83608437, 0.01838356, 0.01862682, 0.01791963, 0.01226628,
        0.0172498 , 0.02080018, 0.01985842, 0.0206185 , 0.01819235],
       [0.02452771, 0.03081482, 0.14634302, 0.68820685, 0.0199675 ,
        0.03976144, 0.01232851, 0.00597061, 0.01715526, 0.01492433],
       [0.04067608, 0.19861075, 0.0572519

## Post-processing

Follow [mewc-predict](https://github.com/zaandahl/mewc-predict/blob/main/src/mewc_predict.py)

In [9]:
with open("../input/train-and-evaluate-terainet/class_list.yaml", "r") as file:
    class_map = yaml.safe_load(file)
class_map

{'1': 'tiger',
 '10': 'bird',
 '2': 'leopard',
 '3': 'black_bear',
 '4': 'other_carnivores',
 '5': 'deer',
 '6': 'wild_boar',
 '7': 'buffalo',
 '8': 'rhino',
 '9': 'elephant'}

In [10]:
inv_class = {v: k for k, v in class_map.items()}
inv_class

{'tiger': '1',
 'bird': '10',
 'leopard': '2',
 'black_bear': '3',
 'other_carnivores': '4',
 'deer': '5',
 'wild_boar': '6',
 'buffalo': '7',
 'rhino': '8',
 'elephant': '9'}

In [11]:
file_paths = img_generator.file_paths
filenames = list(map(lambda x: Path(x).name, file_paths))
labels = list(map(lambda x: Path(x).parent.name, file_paths))

In [12]:
class_ids = sorted(inv_class.values())
class_names = [class_map.get(i, i) for i in class_ids]
pred_df = pd.DataFrame(preds, columns=class_ids)
pred_df.head()

,1,10,2,3,4,5,6,7,8,9
0,0.846743,0.023706,0.011072,0.013728,0.018312,0.015076,0.022011,0.020696,0.011987,0.016670
1,0.007923,0.031212,0.761711,0.023866,0.025573,0.035098,0.029069,0.015093,0.061491,0.008965
2,0.740985,0.067740,0.028803,0.025167,0.036890,0.014406,0.024174,0.019895,0.017727,0.024214
3,0.830365,0.023986,0.014834,0.020671,0.021976,0.022334,0.016183,0.018635,0.015901,0.015116
4,0.726912,0.081801,0.024233,0.030613,0.026575,0.034167,0.020315,0.010139,0.027851,0.017394


In [13]:
file_series = pd.Series(filenames)
label_series = pd.Series(labels)
pred_df.insert(0, "filename", file_series, True)
pred_df.insert(1, "label", label_series, True)
pred_df = pd.melt(
    pred_df,
    id_vars=["filename", "label"],
    value_vars=class_ids,
    var_name="class_id",
    value_name="prob",
)
pred_df["class_name"] = pred_df["class_id"].replace(class_map)
pred_df["class_rank"] = pred_df.groupby("filename")["prob"].rank("average", ascending=False)
pred_df.head(10)

,filename,label,class_id,prob,class_name,class_rank
0,class_1_test2_118-0.jpg,images,1,0.846743,tiger,1.0
1,class_1_test2_135-0.jpg,images,1,0.007923,tiger,10.0
2,class_1_test2_143-0.jpg,images,1,0.740985,tiger,1.0
3,class_1_test2_185-0.jpg,images,1,0.830365,tiger,1.0
4,class_1_test2_19-0.jpg,images,1,0.726912,tiger,1.0
5,class_1_test2_210-0.jpg,images,1,0.836084,tiger,1.0
6,class_1_test2_283-0.jpg,images,1,0.024528,tiger,5.0
7,class_1_test2_3-0.jpg,images,1,0.040676,tiger,5.0
8,class_1_test2_310-0.jpg,images,1,0.847348,tiger,1.0
9,class_1_test2_75-0.jpg,images,1,0.774054,tiger,1.0


In [14]:
pred_df = pred_df[pred_df["class_rank"] == 1.0]
pred_df = pred_df.drop(["label", "class_rank"], axis=1)
pred_df

,filename,class_id,prob,class_name
0,class_1_test2_118-0.jpg,1,0.846743,tiger
2,class_1_test2_143-0.jpg,1,0.740985,tiger
3,class_1_test2_185-0.jpg,1,0.830365,tiger
4,class_1_test2_19-0.jpg,1,0.726912,tiger
5,class_1_test2_210-0.jpg,1,0.836084,tiger
8,class_1_test2_310-0.jpg,1,0.847348,tiger
9,class_1_test2_75-0.jpg,1,0.774054,tiger
21,class_1_test2_135-0.jpg,2,0.761711,leopard
36,class_1_test2_283-0.jpg,3,0.688207,black_bear
37,class_1_test2_3-0.jpg,3,0.595859,black_bear
